# Q Kaggle Free Training

This notebook is bound to the tracked Q + Immaculate session `q-hybrid-harbor-opt-2384cf5-bench-v20`.

It does three concrete things:
- clones the exact repo commit behind the current release surface
- pulls the staged cloud bundle from Hugging Face when a token is present
- runs the bounded Q micro-train plus the Immaculate bundle rebuild if the Kaggle GPU is large enough

Truth boundary:
- this is a supplemental Kaggle notebook lane, not the primary cloud trainer
- it does not claim a Kaggle launch happened unless a separate operator actually starts the notebook
- if GPU memory is too small, the notebook should stop at doctor plus bundle replay

In [ ]:
SESSION_ID = "q-hybrid-harbor-opt-2384cf5-bench-v20"
REPO_URL = "https://github.com/PossumXI/Immaculate.git"
REPO_COMMIT = "33af2569b63480e4292e70c35b573500760df5b1"
HF_DATASET_REPO = "TruLumecreator/immaculate-q-cloud-bundles"
HF_ARCHIVE_PATH = "sessions/q-hybrid-harbor-opt-2384cf5-bench-v20/q-hybrid-harbor-opt-2384cf5-bench-v20-cloud-bundle.tar.gz"
HF_MANIFEST_PATH = "sessions/q-hybrid-harbor-opt-2384cf5-bench-v20/bundle-manifest.json"
SESSION_MANIFEST_RELATIVE = ".training-output/q/sessions/q-hybrid-harbor-opt-2384cf5-bench-v20/hybrid-session.manifest.json"
Q_CONFIG_RELATIVE = ".training-output/q/q-lora-config-harbor-opt-2384cf5-bench-v20.json"
IMMACULATE_BUNDLE_OUTPUT = ".training-output/immaculate/immaculate-training-bundle-q-hybrid-harbor-opt-2384cf5-bench-v20.json"
MICRO_CONFIG_RELATIVE = ".training-output/q/q-hybrid-harbor-opt-2384cf5-bench-v20-micro-config.json"
MICRO_MAX_STEPS = 40
MICRO_MAX_SEQ_LENGTH = 4096
MIN_GPU_MEMORY_GB = 15
RUN_MICRO_TRAIN = True

In [ ]:
import os
import shutil
import subprocess
import sys
import tarfile
from pathlib import Path

def run(command, cwd=None, env=None):
    print("$", " ".join(command))
    return subprocess.run(command, cwd=cwd, env=env, check=True)

def safe_extract(archive_path: Path, target_dir: Path):
    target_dir = target_dir.resolve()
    with tarfile.open(archive_path, "r:gz") as handle:
        for member in handle.getmembers():
            candidate = (target_dir / member.name).resolve()
            if not str(candidate).startswith(str(target_dir)):
                raise ValueError(f"Unsafe tar entry: {member.name}")
        handle.extractall(target_dir)

WORKSPACE = Path("/kaggle/working/immaculate-kaggle")
REPO_ROOT = WORKSPACE / "repo"
WORKSPACE.mkdir(parents=True, exist_ok=True)

if REPO_ROOT.exists():
    shutil.rmtree(REPO_ROOT)

run(["git", "clone", REPO_URL, str(REPO_ROOT)])
run(["git", "checkout", REPO_COMMIT], cwd=str(REPO_ROOT))

In [ ]:
run([sys.executable, "-m", "pip", "install", "--upgrade", "pip"])
run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "huggingface_hub",
        "datasets",
        "transformers",
        "trl",
        "accelerate",
        "peft",
        "bitsandbytes",
        "wandb",
        "unsloth",
    ]
)

In [ ]:
HF_TOKEN = os.environ.get("HF_TOKEN", "").strip()
if not HF_TOKEN:
    raise ValueError("HF_TOKEN is required to pull the staged Q cloud bundle in Kaggle.")

WANDB_API_KEY = os.environ.get("WANDB_API_KEY", "").strip()
WANDB_ENTITY = os.environ.get("WANDB_ENTITY", "").strip()
WANDB_PROJECT = os.environ.get("WANDB_PROJECT", "").strip()

In [ ]:
if HF_DATASET_REPO and HF_ARCHIVE_PATH and HF_MANIFEST_PATH:
    from huggingface_hub import hf_hub_download

    archive_path = Path(
        hf_hub_download(
            repo_id=HF_DATASET_REPO,
            filename=HF_ARCHIVE_PATH,
            repo_type="dataset",
            token=HF_TOKEN,
        )
    )
    manifest_path = Path(
        hf_hub_download(
            repo_id=HF_DATASET_REPO,
            filename=HF_MANIFEST_PATH,
            repo_type="dataset",
            token=HF_TOKEN,
        )
    )
    safe_extract(archive_path, REPO_ROOT)
    print("Archive:", archive_path)
    print("Manifest:", manifest_path)
else:
    print("No staged HF bundle recorded; proceeding with repo-only session files.")

In [ ]:
run(
    [
        sys.executable,
        "training/q/run_q_training_session.py",
        "--doctor",
        "--session",
        SESSION_MANIFEST_RELATIVE,
    ],
    cwd=str(REPO_ROOT),
)

run(
    [
        sys.executable,
        "training/immaculate/build_immaculate_training_bundle.py",
        "--output",
        IMMACULATE_BUNDLE_OUTPUT,
    ],
    cwd=str(REPO_ROOT),
)

In [ ]:
from pathlib import Path

kaggle_output_root = Path("/kaggle/working/immaculate-q-runs") / SESSION_ID
kaggle_output_root.mkdir(parents=True, exist_ok=True)
micro_output_dir = kaggle_output_root / "q-kaggle-free"

run(
    [
        sys.executable,
        "training/q/build_q_colab_micro_config.py",
        "--config",
        Q_CONFIG_RELATIVE,
        "--output",
        MICRO_CONFIG_RELATIVE,
        "--max-steps",
        str(MICRO_MAX_STEPS),
        "--max-seq-length",
        str(MICRO_MAX_SEQ_LENGTH),
        "--gradient-accumulation-steps",
        "8",
        "--output-dir",
        str(micro_output_dir),
    ] + (["--disable-wandb"] if not WANDB_API_KEY else []),
    cwd=str(REPO_ROOT),
)

In [ ]:
import torch

gpu_visible = torch.cuda.is_available()
total_gpu_memory_gb = 0.0
if gpu_visible:
    props = torch.cuda.get_device_properties(0)
    total_gpu_memory_gb = props.total_memory / (1024 ** 3)

print({"gpu_visible": gpu_visible, "total_gpu_memory_gb": round(total_gpu_memory_gb, 2)})

if RUN_MICRO_TRAIN and gpu_visible and total_gpu_memory_gb >= MIN_GPU_MEMORY_GB:
    env = dict(os.environ)
    if WANDB_API_KEY:
        env["WANDB_API_KEY"] = WANDB_API_KEY
    if WANDB_ENTITY:
        env["WANDB_ENTITY"] = WANDB_ENTITY
    if WANDB_PROJECT:
        env["WANDB_PROJECT"] = WANDB_PROJECT
    run(
        [
            sys.executable,
            "training/q/train_q_lora_unsloth.py",
            "--config",
            MICRO_CONFIG_RELATIVE,
        ],
        cwd=str(REPO_ROOT),
        env=env,
    )
else:
    print("Skipping micro-train because the available Kaggle GPU is too small or not visible.")